### Load in the Model (DeepSeek-R1-Distill-Llama-8B)

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import os
import re
import pandas as pd
from tqdm import tqdm
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-R1-Distill-Llama-8B")
model = AutoModelForCausalLM.from_pretrained("deepseek-ai/DeepSeek-R1-Distill-Llama-8B", device_map="cuda", dtype=torch.bfloat16)
print(model.device)
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

c:\Users\Faruk\miniconda3\envs\algoverse\Lib\site-packages\transformers\modeling_utils.py:5198: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:39.)
  _ = torch.empty(int(byte_count // 2), dtype=torch.float16, device=device, requires_grad=False)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

cuda:0
Greetings! I'm DeepSeek-R1, an artificial intelligence assistant created by DeepSeek. I'm at your service and would be delighted to assist you with any inquiries or tasks you may have.
</think>


### Loading the Datasets

In [3]:
# Load in the Factual Datasets
F0_train, F0_test = pd.read_csv("../dataset/F0_train.csv")[["statement", "label"]], pd.read_csv("../dataset/F0_test.csv")[["statement", "label"]]
F1_train, F1_test = pd.read_csv("../dataset/F1_train.csv")[["statement", "label"]], pd.read_csv("../dataset/F1_test.csv")[["statement", "label"]]
F2_train, F2_test = pd.read_csv("../dataset/F2_train.csv"), pd.read_csv("../dataset/F2_test.csv")
F3_train, F3_test = pd.read_csv("../dataset/F3_train.csv"), pd.read_csv("../dataset/F3_test.csv")
F4_train, F4_test = pd.read_csv("../dataset/F4_train.csv"), pd.read_csv("../dataset/F4_test.csv")
F5_train, F5_test = pd.read_csv("../dataset/F5_train.csv"), pd.read_csv("../dataset/F5_test.csv")

# Load in the Arithmatic Statements
A1_train, A1_test = pd.read_csv("../dataset/A1_train.csv"), pd.read_csv("../dataset/A1_test.csv")
A2_train, A2_test = pd.read_csv("../dataset/A2_train.csv"), pd.read_csv("../dataset/A2_test.csv")
A3_train, A3_test = pd.read_csv("../dataset/A3_train.csv"), pd.read_csv("../dataset/A3_test.csv")


In [4]:
datasets = {
    "F0_train": F0_train, "F0_test": F0_test,
    "F1_train": F1_train, "F1_test": F1_test,
    "F2_train": F2_train, "F2_test": F2_test,
    "F3_train": F3_train, "F3_test": F3_test,
    "F4_train": F4_train, "F4_test": F4_test,
    "F5_train": F5_train, "F5_test": F5_test,
    "A1_train": A1_train, "A1_test": A1_test,
    "A2_train": A2_train, "A2_test": A2_test,
    "A3_train": A3_train, "A3_test": A3_test,
}

for name, df in datasets.items():
    print(f"{name}: {len(df)}")


F0_train: 1194
F0_test: 512
F1_train: 1194
F1_test: 512
F2_train: 1194
F2_test: 512
F3_train: 1398
F3_test: 600
F4_train: 1394
F4_test: 598
F5_train: 1383
F5_test: 593
A1_train: 700
A1_test: 300
A2_train: 700
A2_test: 300
A3_train: 700
A3_test: 300


### Funtions to Generate Activations

In [5]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

# DeepSeek-R1-Distill always reasons inside <think>...</think> on its own — we don't need
# to force that structure via instructions, just guide *what* the reasoning should cover.
# Keep this loose: an over-rigid "one line per city, no summary" instruction fights the
# model's native reasoning style (that fought Llama-3.1-8B into unreliable arithmetic).
#
# A literal marker phrase ("FINAL VERDICT:") did NOT work: across every sampled trace,
# the model ignored the instruction to place that exact marker inside <think> and instead
# closed </think> with its own natural conclusion sentence, then invented its own
# "Final Verdict"-style heading in the *answer* section.
#
# We still ask it to reserve True/False for its conclusion (reduces incidental noise),
# but we don't rely on that alone: a forward keyword search fails on its own too, because
# every trace OPENS with "...trying to figure out whether the statement '...' is true or
# false", which is a false-positive hit on the very first sentence, long before any real
# reasoning. So we anchor backward from </think> instead, restrict the search to only the
# last few lines before it (the "commit region"), and match a wider set of phrasings
# DeepSeek actually uses to commit ("is incorrect", "does not hold", "does not match",
# etc.), not just bare true/false.
# COT_INSTRUCTIONS = (
#     "Work through the statement above before answering. State the facts you need — for "
#     "each city named, where it actually is; for each arithmetic sub-expression, its "
#     "actual value — then explicitly compute or count what the statement claims, and "
#     "compare. Reserve the words 'True' and 'False' only for stating your conclusion — do "
#     "not use either word anywhere else in your reasoning. Once you have compared the "
#     "computed value against the statement's claim, write your conclusion as exactly one "
#     "line containing only the word True or False. After you are done reasoning, write "
#     "exactly one line: \nAnswer: Yes or Answer: No"
# )

# COT_INSTRUCTIONS = (
#     "Work through the statement above before answering.\n\n"
#     "First, for each city named, state its location as a plain sentence in exactly this "
#     "form: 'CITY is in COUNTRY.' For example: 'Istanbul is in Turkey.' Do not write "
#     "anything after the country name — no commas, dashes, checkmarks, or words like "
#     "'correct', 'yes', or 'right'. Just the city, 'is in', the country, and a period.\n\n"
#     "After listing every city this way, count how many fall in each country the statement "
#     "mentions, and compare those counts to what the statement claims.\n\n"
#     "Do not use the words True, False, correct, incorrect, accurate, or inaccurate at any "
#     "point while reasoning. Reserve them for your conclusion only.\n\n"
#     "End your reasoning with exactly one line, either:\n"
#     "The statement is true.\n"
#     "or\n"
#     "The statement is false.\n"
#     "Then write </think>. After </think>, on its own line, write exactly one of:\n"
#     "Answer: Yes\n"
#     "or\n"
#     "Answer: No"
# )
COT_INSTRUCTIONS = (
    "Work through the statement above before answering. State the facts you need — for "
    "each city named, where it actually is; for each arithmetic sub-expression, its "
    "actual value — then explicitly compute or count what the statement claims, and "
    "compare.\n\n"
    "Do not use the words True, False, correct, incorrect, accurate, or inaccurate while "
    "reasoning. Reserve them for your conclusion only.\n\n"
    "End your reasoning with exactly one line, either:\n"
    "The statement is true.\n"
    "or\n"
    "The statement is false.\n"
    "Then write </think>. After </think>, on its own line, write exactly one of:\n"
    "Answer: Yes\n"
    "or\n"
    "Answer: No"
)


# Ordered roughly by how DeepSeek commits. All matched case-insensitively as whole words /
# phrases. Add any phrasing you see slip through during an audit_cuts() pass. Bare
# true/false go last so the more specific phrase forms win when both are present on a line.
#
# NOTE: earlier versions of this list also included bare-transition patterns like
# r"\btherefore,?\s+the\s+statement\b" and r"\bso\s+the\s+(?:answer|statement)\b".
# Those were a real bug: "Therefore, the statement" doesn't itself reveal anything —
# the actual reveal can come several words or even a sentence later ("...is saying 0
# are in Tanzania when in fact 1 is. That makes the entire statement false."). Matching
# on the bare connective cut way too early, discarding legitimate reasoning between the
# connective and the real reveal word. Every pattern below requires an actual reveal
# word (true/false/correct/incorrect/accurate/hold/match) to be present, so cutting
# only ever happens right before genuine information leaks, never before a mere
# transition phrase.
LEAD_IN = r"(?:is|to\s+be|seems(?:\s+to\s+be)?)"

VERDICT_PATTERNS = [
    rf"\b{LEAD_IN}\s+((?:in)?correct)\b",
    rf"\b{LEAD_IN}\s+(?:not\s+)?(true)\b",
    rf"\b{LEAD_IN}\s+(false)\b",
    rf"\b{LEAD_IN}\s+((?:in)?accurate)\b",
    r"\bdoes(?:n't| not)\s+(hold)\b",
    r"\bdoes(?:n't| not)\s+(match)\b",
    r"\b(?:does(?:n't| not)\s+)?(aligns?)\s+(?:perfectly\s+)?with\b",
    r"\bstatement\s+is\s+(?:therefore\s+)?(true|false|correct|incorrect)\b",
    r"\b((?:in)?correct)\b",   # bare fallback, mirrors true/false
    r"\b((?:in)?accurate)\b",  # bare fallback
    r"(true)\b",
    r"(false)\b",
]

VERDICT_RE = re.compile("|".join(VERDICT_PATTERNS), re.IGNORECASE)
ANCHOR_RE = re.compile(
    r"\b(?:therefore|thus|hence|in\s+summary|in\s+conclusion|to\s+conclude|overall)\b",
    re.IGNORECASE
)

# How many lines above </think> count as the "commit region". DeepSeek's conclusion is
# 1-3 lines; 4 gives margin without letting the opening restatement back into scope.
COMMIT_REGION_LINES = 4

# </think>'s exact token id sequence for this tokenizer, computed once. Used to anchor
# the backward search at the token-id level (robust to skip_special_tokens stripping it
# from decoded text, if it's registered as a special token).
THINK_CLOSE_IDS = tokenizer.encode("</think>", add_special_tokens=False)


def _find_think_close(ids, think_close_ids):
    """Token index where </think> starts, searching from the end since it's near there."""
    k = len(think_close_ids)
    for j in range(len(ids) - k, -1, -1):
        if ids[j:j + k] == think_close_ids:
            return j
    return None


def _char_to_token_idx(tokenizer, ids, upper_tok, target_char):
    lo, hi = 0, upper_tok
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if len(tokenizer.decode(ids[:mid], skip_special_tokens=True)) < target_char:
            lo = mid
        else:
            hi = mid - 1
    return lo

def _mask_parens(text):
    """Blank out the contents of any (...) spans (same length, so character
    offsets elsewhere stay valid) before searching for a verdict match.
    Targets parenthetical asides like "(which seems correct)" that comment
    on a sub-claim without being the actual final verdict, without the
    broader over-skipping problem the anchor-based approach had."""
    return re.sub(r"\([^()]*\)", lambda m: " " * len(m.group(0)), text)

def find_pre_verdict_cut(tokenizer, ids, think_close_ids=THINK_CLOSE_IDS):
    """
    ids: 1D list of token ids for the generated continuation only (no prompt).

    Returns a token index into `ids` to cut at (activation = last token of ids[:cut]),
    positioned just before the model commits to a verdict.

    Strategy:
      1. Find </think>. If absent (truncated generation), return len(ids).
      2. Look only at the last COMMIT_REGION_LINES lines before </think> — this keeps
         the opening restatement ("...is true or false") out of scope entirely.
      3. First verdict-phrase match in that region -> cut just before it.
      4. No match in region -> cut at the newline before </think> (best structural
         fallback; a sign VERDICT_PATTERNS needs to be widened further).
    """
    ids = list(ids)
    n = len(ids)
    if n == 0:
        return 0

    close_at = _find_think_close(ids, think_close_ids)
    if close_at is None:
        return n

    pre = tokenizer.decode(ids[:close_at], skip_special_tokens=True)

    pre_lines = pre.split("\n")
    region_start_char = len("\n".join(pre_lines[:-COMMIT_REGION_LINES]))
    if region_start_char > 0:
        region_start_char += 1  # step over the newline separating the region from the rest

    region_text = pre[region_start_char:]
    search_text = _mask_parens(region_text)
    # anchor_matches = list(ANCHOR_RE.finditer(region_text))
    # m_anchor = anchor_matches[-1] if anchor_matches else None

    # if m_anchor:
    #     m = VERDICT_RE.search(region_text[m_anchor.end():])
    #     if m is not None:
    #         target_char = region_start_char + m_anchor.end() + m.start(m.lastindex)
    #     else:
    #         m = VERDICT_RE.search(region_text)
    #         if m is not None:
    #             target_char = region_start_char + m.start(m.lastindex)
    # else:

    m = VERDICT_RE.search(search_text)

    if m is not None:
        target_char = region_start_char + m.start(m.lastindex)
        return _char_to_token_idx(tokenizer, ids, close_at, target_char)


    # Fallback: no known verdict phrasing found in the commit region. Cut at the
    # newline before </think> rather than leaking all the way up to it.
    last_nl = pre.rfind("\n")
    if last_nl == -1:
        return close_at
    return _char_to_token_idx(tokenizer, ids, close_at, last_nl)


# Format-only demos: unrelated dummy examples per task type, showing the expected <think>
# structure — facts, then an explicit computed/counted value, then a bare "True"/"False"
# conclusion line, and nothing else. No other evaluative language at all (no "matches",
# "contradicts", "fails", "correct", etc.) — only raw facts, numbers, and the reserved
# verdict word, so nothing else in the demo hints at a verdict early. Each task gets one
# label-true and one label-false demo so the pair itself carries no directional bias.
FEWSHOT_DEMOS = {
    "F0": [
        (
            "The city of Lyon is in France.",
            "<think>\nLyon is in France.\nTrue\n</think>\n\nAnswer: Yes",
        ),
        (
            "The city of Lyon is in Germany.",
            "<think>\nLyon is in France.\nFalse\n</think>\n\nAnswer: No",
        ),
    ],
    "F1": [
        (
            "The city of Lyon is not in Germany.",
            "<think>\nLyon is in France.\nTrue\n</think>\n\nAnswer: Yes",
        ),
        (
            "The city of Lyon is not in France.",
            "<think>\nLyon is in France.\nFalse\n</think>\n\nAnswer: No",
        ),
    ],
    "F2": [
        (
            "It is the case both that The city of Lyon is in France. and that The city of Osaka is in Japan..",
            "<think>\nLyon is in France.\nOsaka is in Japan.\nTrue\n</think>\n\nAnswer: Yes",
        ),
        (
            "It is the case both that The city of Lyon is in France. and that The city of Osaka is in Germany..",
            "<think>\nLyon is in France.\nOsaka is in Japan.\nFalse\n</think>\n\nAnswer: No",
        ),
    ],
    "F3": [
        (
            "Exactly 1 of the following cities are in Germany: Lyon, Berlin.",
            "<think>\nLyon is in France.\nBerlin is in Germany.\nCount of cities in Germany: 1."
            "\nTrue\n</think>\n\nAnswer: Yes",
        ),
        (
            "Exactly 1 of the following cities are in Germany: Lyon, Marseille.",
            "<think>\nLyon is in France.\nMarseille is in France.\nCount of cities in Germany: 0."
            "\nFalse\n</think>\n\nAnswer: No",
        ),
    ],
    "F4": [
        (
            "Exactly 2 of the following cities are in Germany: Lyon, Berlin, Osaka, Munich, Madrid.",
            "<think>\nLyon is in France.\nBerlin is in Germany.\nOsaka is in Japan.\nMunich is in Germany."
            "\nMadrid is in Spain.\nCount of cities in Germany: 2.\nTrue\n</think>\n\nAnswer: Yes",
        ),
        (
            "Exactly 3 of the following cities are in Germany: Lyon, Berlin, Osaka, Munich, Madrid.",
            "<think>\nLyon is in France.\nBerlin is in Germany.\nOsaka is in Japan.\nMunich is in Germany."
            "\nMadrid is in Spain.\nCount of cities in Germany: 2.\nFalse\n</think>\n\nAnswer: No",
        ),
    ],
    "F5": [
        (
            "Exactly 2 of the following cities are in France and 2 in Japan: "
            "Lyon, Osaka, Berlin, Marseille, Kyoto, Madrid.",
            "\nLyon is in France.\nOsaka is in Japan.\nBerlin is in Germany.\nMarseille is in France."
            "\nKyoto is in Japan.\nMadrid is in Spain.\nCount of cities in France: 2."
            "\nCount of cities in Japan: 2.\nTherefore the statement is true.\n\nAnswer: Yes",
        ),
        (
            "Exactly 2 of the following cities are in France and 1 in Japan: "
            "Lyon, Osaka, Berlin, Marseille, Kyoto, Madrid.",
            "\nLyon is in France.\nOsaka is in Japan.\nBerlin is in Germany.\nMarseille is in France."
            "\nKyoto is in Japan.\nMadrid is in Spain.\nCount of cities in France: 2."
            "\nCount of cities in Japan: 2.\nTherefore the statement is false.\n\nAnswer: No",
        ),
    ],
    "A1": [
        (
            "23 - 8 = 15",
            "<think>\n23 - 8 = 15.\nTrue\n</think>\n\nAnswer: Yes",
        ),
        (
            "23 - 8 = 14",
            "<think>\n23 - 8 = 15.\nFalse\n</think>\n\nAnswer: No",
        ),
    ],
    "A2": [
        (
            "(9 + 3) * 4 = 48",
            "<think>\n9 + 3 = 12.\n12 * 4 = 48.\nTrue\n</think>\n\nAnswer: Yes",
        ),
        (
            "(9 + 3) * 4 = 47",
            "<think>\n9 + 3 = 12.\n12 * 4 = 48.\nFalse\n</think>\n\nAnswer: No",
        ),
    ],
    "A3": [
        (
            "(6 + 2) * (10 / 5) = 16",
            "<think>\n6 + 2 = 8.\n10 / 5 = 2.\n8 * 2 = 16.\nTrue\n</think>\n\nAnswer: Yes",
        ),
        (
            "(6 + 2) * (10 / 5) = 20",
            "<think>\n6 + 2 = 8.\n10 / 5 = 2.\n8 * 2 = 16.\nFalse\n</think>\n\nAnswer: No",
        ),
    ],
}


# With instructions and few shot
def generate_activations_cot_with_fewshot(model, statements, layer, task, with_chat_template=True, batch_size=16, verbose=False, save_datasets=False):
    final_token_activations = []

    statements = list(statements)
    fewshot_pairs = FEWSHOT_DEMOS[task.split("_")[0]]  # [(true-labeled statement, response), (false-labeled statement, response)]

    for i in tqdm(range(0, len(statements), batch_size)):
        statements_temp = statements[i: i+batch_size]
        if with_chat_template:
            messages = [
                [
                    *[
                        msg
                        for demo_statement, demo_response in fewshot_pairs
                        for msg in (
                            {"role": "user", "content": f"{demo_statement}\n\n{COT_INSTRUCTIONS}"},
                            {"role": "assistant", "content": demo_response},
                        )
                    ],
                    {"role": "user", "content": f"{s}\n\n{COT_INSTRUCTIONS}"},
                    {"role": "assistant", "content": "<think>\n"},  # prefill: force continuation inside <think>
                ]
                for s in statements_temp
            ]
            inputs = tokenizer.apply_chat_template(
                messages,
                add_generation_prompt=False,
                continue_final_message=True,  # don't close/re-open a turn after our prefill
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
                padding=True,
            ).to(model.device)
        else:
            demo_block = "\n\n".join(
                f"{demo_statement}\n\n{COT_INSTRUCTIONS}\n\n{demo_response}"
                for demo_statement, demo_response in fewshot_pairs
            )
            prompts = [
                f"{demo_block}\n\n{s}\n\n{COT_INSTRUCTIONS}\n\n<think>\n"
                for s in statements_temp
            ]
            inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)

        with torch.no_grad():
            gen_ids = model.generate(**inputs, max_new_tokens=1000, do_sample=False,
                                    repetition_penalty=1.1,
                                    stop_strings=["Answer: Yes", "Answer: No"],
                                    tokenizer=tokenizer)

        # generate() only returns token IDs, not hidden states, and the batch is padded
        # (left-padding on the prompt, right-padding on generation for rows that finished
        # early) so a shared `[:, -1, :]` index would often land on a pad token instead of
        # real content. For each row: isolate its real (unpadded) generated continuation,
        # find the automated pre-verdict cut point (find_pre_verdict_cut, anchored
        # backward from </think>), then do a separate forward pass over (real prompt +
        # reasoning up to that cut) to get a clean activation that predates the verdict.
        for row_idx, row in enumerate(gen_ids):
            prompt_len = int((inputs["input_ids"][row_idx] != tokenizer.pad_token_id).sum())

            real_ids_full = row[row != tokenizer.pad_token_id].unsqueeze(0)
            gen_part = real_ids_full[0, prompt_len:]
            gen_part_list = gen_part.tolist()
            cut = find_pre_verdict_cut(tokenizer, gen_part_list)

            if verbose:
                full_text = tokenizer.decode(gen_part_list, skip_special_tokens=True)
                extracted_text = tokenizer.decode(gen_part_list[:cut], skip_special_tokens=True)
                print(full_text)
                print("===== EXTRACTED (this is what the activation is taken from) =====")
                print(extracted_text)
                print("---------------------------")

            real_ids = real_ids_full[:, :prompt_len + cut]
            with torch.no_grad():
                fwd_out = model(real_ids, output_hidden_states=True)
            final_token_activations.append(fwd_out.hidden_states[layer][:, -1, :].cpu())

    return torch.cat(final_token_activations, dim=0)

# Only with Instructions
def generate_activations_cot_without_fewshot(model, statements, layer, with_chat_template=True, batch_size=16, verbose=False):
    final_token_activations = []
   
    statements = list(statements)

    for i in tqdm(range(0, len(statements), batch_size)):
        statements_temp = statements[i: i+batch_size]
        if with_chat_template:
            messages = [
                [
                    {"role": "user", "content": f"{s}\n\n{COT_INSTRUCTIONS}"},
                    {"role": "assistant", "content": "<think>\n"},  # prefill: force continuation inside <think>
                ]
                for s in statements_temp
            ]
            inputs = tokenizer.apply_chat_template(
                messages,
                add_generation_prompt=False,
                continue_final_message=True,  # don't close/re-open a turn after our prefill
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
                padding=True,
            ).to(model.device)
        else:
            prompts = [
                f"{s}\n\n{COT_INSTRUCTIONS}\n\n<think>\n"
                for s in statements_temp
            ]
            inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)

        with torch.no_grad():
            gen_ids = model.generate(**inputs, max_new_tokens=1000, do_sample=False,
                                    repetition_penalty=1.1,
                                    stop_strings=["Answer: Yes", "Answer: No"],
                                    tokenizer=tokenizer)
            
        for row_idx, row in enumerate(gen_ids):
            prompt_len = int((inputs["input_ids"][row_idx] != tokenizer.pad_token_id).sum())

            real_ids_full = row[row != tokenizer.pad_token_id].unsqueeze(0)
            gen_part = real_ids_full[0, prompt_len:]
            gen_part_list = gen_part.tolist()
            cut = find_pre_verdict_cut(tokenizer, gen_part_list)

            if verbose:
                full_text = tokenizer.decode(gen_part_list, skip_special_tokens=True)
                extracted_text = tokenizer.decode(gen_part_list[:cut], skip_special_tokens=True)
                print(full_text)
                print("===== EXTRACTED (this is what the activation is taken from) =====")
                print(extracted_text)
                print("---------------------------")

            real_ids = real_ids_full[:, :prompt_len + cut]
            with torch.no_grad():
                fwd_out = model(real_ids, output_hidden_states=True)
            final_token_activations.append(fwd_out.hidden_states[layer][:, -1, :].cpu())


    return torch.cat(final_token_activations, dim=0)

In [10]:
def strip_special_tokens(text, tokenizer=None):
    text = re.sub(r"<｜[^｜]*｜>", "", text)
    text = text.replace("<think>", "").replace("</think>", "")
    return text.strip()

# Loading in the old Few-Shot Methodology
F5_train_generations = pd.read_csv("F5_train.csv")

for row_idx, row in F5_train_generations.head(100).iterrows():
    print(row["generated_statement_texts"].split("<think>")[1])
    print("===== EXTRACTED (this is what the activation is taken from) =====")
    print(row["extracted_statement_texts"].split("<think>")[1])
    print("---------------------------")



Okay, so I'm trying to figure out if the statement "Exactly 5 of the following cities are in the United Arab Emirates and 0 in Tanzania: Curitiba, Al Mawsil al Jadidah, Izhevsk, Vadodara, Zanzibar, Thembisa." is true or false. Let me break this down step by step.

First, I need to identify which cities are actually in the UAE and which are in Tanzania. The list given is Curitiba, Al Mawsil al Jadidah, Izhevsk, Vadodara, Zanzibar, and Thembisa.

Starting with Curitiba: I know that Curitiba is a city in Brazil. So, it's definitely not in the UAE or Tanzania.

Next up is Al Mawsil al Jadidah. This name sounds familiar, but I'm not entirely sure where it is. I think it might be in Saudi Arabia. If I recall correctly, Al Mawsil al Jadidah is also known as Al Mecca, which is a holy city in Saudi Arabia. So, that means it's not in the UAE either.

Moving on to Izhevsk: I believe Izhevsk is a city in Russia. It's the capital of the Udmurt Republic. Since Russia isn't the UAE or Tanzania, Izhe

In [9]:
activations = generate_activations_cot_without_fewshot(model, F5_train["statement"][:100], 16, with_chat_template=True, batch_size=12, verbose=True)

  0%|          | 0/9 [00:00<?, ?it/s]

Okay, so I have this problem here that says exactly 5 out of the listed cities are in the UAE and none are in Tanzania. The cities given are Curitiba, Al Mawsil al Jadidah, Izhevsk, Vadodara, Zanzibar, and Thembisa. My task is to figure out if this statement is true or false by checking each city's location.

First, I'll list all the cities and their supposed countries:

1. Curitiba – I think this is in Brazil because I remember hearing about Curitiba being a major city there. So, not in UAE or Tanzania.

2. Al Mawsil al Jadidah – This name sounds Arabic. I believe it's also known as Mosul, which is in Iraq. So again, not in UAE or Tanzania.

3. Izhevsk – I'm pretty sure this is in Russia. It's a city I've heard of in the Volga region. So, not in UAE or Tanzania.

4. Vadodara – This one I'm less certain about. I think it's an Indian city, maybe in Gujarat state. So, not in UAE or Tanzania.

5. Zanzibar – I know Zanzibar is an island off the coast of Tanzania. Wait, but the statement sa

 11%|█         | 1/9 [00:37<04:57, 37.20s/it]

Okay, so I have this problem here that says exactly 3 out of the listed cities are in Tanzania and none are in Brazil. The cities given are Manaus, Guarulhos, Dodoma, Arusha, Dombivali, and Feira de Santana. I need to figure out if this statement is true or false by checking each city's location.

First, I'll list all the cities and determine which country they're in. Let me go one by one:

1. **Manaus**: I think Manaus is a city in Brazil. It's the capital of Amazonas state, right? So that's definitely Brazil.

2. **Guarulhos**: Hmm, Guarulhos sounds familiar. I believe it's a city in Brazil as well. Maybe in São Paulo state? Yeah, I'm pretty sure it's in Brazil.

3. **Dodoma**: This name rings a bell. I recall that Dodoma is the capital of Tanzania. So that's in Tanzania.

4. **Arusha**: Arusha is another Tanzanian city. I think it's near Mount Kilimanjaro and is known for tourism. Definitely in Tanzania.

5. **Dombivali**: Wait, Dombivali... That doesn't sound like a place I'm very 

 22%|██▏       | 2/9 [01:09<03:58, 34.14s/it]

...

 78%|███████▊  | 7/9 [04:17<01:14, 37.03s/it]

Okay, so I have this problem here that says exactly 5 out of the listed cities are in Nigeria and 1 in Zimbabwe. The cities given are Bulawayo, Katsina, Ibadan, Abuja, Zaria, and Ilorin. My task is to figure out if this statement is true or false by checking each city's location.

First, I'll list all the cities: Bulawayo, Katsina, Ibadan, Abuja, Zaria, Ilorin. That's six cities in total.

Now, I need to determine how many of these are in Nigeria and how many in Zimbabwe. The statement claims exactly 5 are in Nigeria and 1 in Zimbabwe. So, I should check each city one by one.

Starting with Bulawayo. I remember that Bulawayo is a city in Zimbabwe. It's actually the second largest city there. So that's one city accounted for in Zimbabwe.

Next up is Katsina. I think Katsina is in Nigeria. Let me confirm. Yes, Katsina is a city in the northern part of Nigeria, known for its cultural significance. So that's one for Nigeria.

Ibadan is next. I'm pretty sure Ibadan is a major city in Nigeri

 89%|████████▉ | 8/9 [04:59<00:38, 38.47s/it]

Okay, so I have this problem here that says exactly 5 out of the listed cities are in Russia and 1 in Iran. The cities given are Barnaul, Astrakhan, Irkutsk, Orenburg, Karaj, and Krasnoyarsk. My task is to figure out if this statement is true or false by checking each city's location.

First, I'll list all the cities: Barnaul, Astrakhan, Irkutsk, Orenburg, Karaj, Krasnoyarsk. That's six cities total.

Now, I need to determine how many of these are in Russia and how many are in Iran. The statement claims exactly 5 are in Russia and 1 in Iran. So, I should check each city one by one.

Starting with Barnaul. I think Barnaul is a city in Russia. It's in the Altai region, right? So that's definitely Russia.

Next, Astrakhan. I'm pretty sure Astrakhan is also in Russia. It's a major port city on the Volga River. So that's another Russian city.

Moving on to Irkutsk. Irkutsk is another Russian city. It's located in Siberia, known for its history and architecture. So that's three so far.

Oren

100%|██████████| 9/9 [05:28<00:00, 36.54s/it]


In [6]:
activations = generate_activations_cot_with_fewshot(model, F5_train["statement"][:100], 16, "F5_train", with_chat_template=True, batch_size=10, verbose=True)

  0%|          | 0/10 [00:00<?, ?it/s]

Alright, let's tackle this problem step by step. The statement says that exactly 5 out of the listed cities are in the United Arab Emirates (UAE) and none are in Tanzania. Let me go through each city one by one to verify their locations.

First, I'll list all the cities mentioned: Curitiba, Al Mawsil al Jadidah, Izhevsk, Vadodara, Zanzibar, and Thembisa. That makes six cities in total.

Starting with Curitiba. I know Curitiba is a city in Brazil. So, it's definitely not in the UAE or Tanzania.

Next up is Al Mawsil al Jadidah. This name sounds familiar, but I'm not entirely sure where it is. I think it might be a city in Iraq. If that's correct, then it's also not in the UAE or Tanzania.

Moving on to Izhevsk. I believe Izhevsk is a city in Russia. Again, not in the UAE or Tanzania.

Vadodara is next. Vadodara is a city in India, so that's another one not in the UAE or Tanzania.

Now, Zanzibar. I recall that Zanzibar is an island off the coast of Tanzania. In fact, Zanzibar is part of 

 10%|█         | 1/10 [00:38<05:43, 38.14s/it]

Okay, so I'm trying to figure out if this statement is true or false. The statement says that exactly 1 of the listed cities is in Côte d'Ivoire and exactly 4 are in Ukraine. The cities mentioned are Odesa, Sevastopol, Cangzhou, Bouake, Dnipro, and Kryvyy Rih.

First, I need to determine which of these cities are actually located in Côte d'Ivoire and which are in Ukraine. Let me go through each city one by one.

Starting with Odesa: I know that Odesa is a city in Ukraine. It's an important port city, so that's definitely in Ukraine.

Next up is Sevastopol. From what I remember, Sevastopol is also in Ukraine. It's a major coastal city and has historical significance there.

Now, Cangzhou: This name sounds familiar, but I'm not entirely sure where it is. I think it might be in China because the name doesn't ring a bell as a Ukrainian city. To confirm, I'll recall that Cangzhou is a city in Guangdong Province in southern China. So, it's not in Côte d'Ivoire or Ukraine.

Moving on to Bouak

 20%|██        | 2/10 [01:11<04:41, 35.22s/it]

...

 80%|████████  | 8/10 [04:51<01:12, 36.25s/it]

Alright, let's tackle this problem step by step. The statement says that exactly 3 of the listed cities are in Bangladesh and 1 in Jordan. The cities given are Gazipur, Denpasar, Huizhou, Rajshahi, Irbid, and Mirpur Model Thana.

First, I'll list out each city and determine their locations:

1. **Gazipur**: This is a city in Bangladesh.
2. **Denpasar**: I know that Denpasar is the capital of Bali in Indonesia, so it's definitely not in Bangladesh.
3. **Huizhou**: Huizhou is a city in China, specifically in the Henan province. So, it's not in Bangladesh.
4. **Rajshahi**: Rajshahi is another city in Bangladesh.
5. **Irbid**: Irbid is located in Jordan.
6. **Mirpur Model Thana**: This is an area within Dhaka, which is the capital of Bangladesh.

Now, let's count how many cities are in Bangladesh based on the information gathered:

- Gazipur (Bangladesh)
- Rajshahi (Bangladesh)
- Mirpur Model Thana (Bangladesh)

That makes 3 cities in Bangladesh. 

Next, checking the part about Jordan:

- 

 90%|█████████ | 9/10 [05:26<00:35, 35.96s/it]

Okay, so I have this problem here that says exactly 4 out of these cities are in Egypt and 1 in Kenya. The cities listed are Al Fayyum, Al Mahallah al Kubra, Thiruvananthapuram, Port Said, Nairobi, and Suez. Hmm, let me try to figure this out step by step.

First, I need to determine which of these cities are actually located in Egypt and which ones are in Kenya. I'm not entirely sure about all of them, but I'll do my best.

Starting with Al Fayyum. I think Al Fayyum is a city in Egypt. I've heard of it being an archaeological site, so that makes sense.

Next up is Al Mahallah al Kubra. That name sounds Egyptian to me. I believe it's a city known for its historical significance, probably in Upper Egypt.

Thiruvananthapuram is a bit tricky. I know that Trivandrum is a major city in India, specifically in Kerala state. So, that's definitely not in Egypt or Kenya.

Port Said is another one. I remember hearing that Port Said is a significant port city along the Nile River in Egypt. So, tha

100%|██████████| 10/10 [05:59<00:00, 35.98s/it]


### Evaluation

In this notebook I tried out different ways to prompt the model while retreiving CoT generations from it in order to optimize it such that the determenistic regex-based pre-judgement token extraction mechanism works with a high success rate. I tried three different methods, 1) With few-shot learning and CoT instructions, but the few-shot learning examplars do not have reasoning chains 2) With few-shot learning and CoT instructions, but the few-shot learning examplars do have reasoning chains 3) With only CoT instructions. As a result I saw that the pure-instruction based one was the best out of the three, so our project will adopt the 3rd methodology for both token size concerns and not to over-guide model generation (creating a possible biased set of outputs).

Across 100 held-out statements each, the instructions-only method achieved the highest clean extraction rate (95%), versus 87% for few-shot with reasoning-free exemplars and 85% for few-shot with full reasoning chains. The gap exceeds run-to-run variance from batched-greedy decoding (±~3%), so instructions-only is not merely tied — it is modestly better while also using ~800 fewer tokens per prompt and avoiding exemplar-induced guidance of the reasoning distribution. Notably, the reasoning-chain exemplars performed worst: they reintroduced meta-commentary such as "my initial assessment seems correct," which shifts the cut off the statement-level verdict.

Extraction correctness was assessed by manual per-sample audit against a fixed rubric (a cut is correct when it lands on the pre-verdict token of the model's whole-statement conclusion); the residual failures fall into a small, consistent taxonomy across all three conditions — per-city "City: Correct" checklist lines, self-assessment beats, sub-part "aligns with" statements, and unmatched commit phrasings like "accurately reflects" — indicating the errors are model-intrinsic rather than prompt-specific.